# 电化学反应的 EDBO+ 贝叶斯优化全流程

本 notebook 参考 `examples/tutorials/1_CLI_example.ipynb` 的 CLI 工作流，并使用 `experiment_conditions.md` 中给出的电化学条件空间。流程包括：

1. 定义组合条件空间并生成 CSV；
2. 在没有实验数据时让 EDBO+ 推荐初始实验；
3. 录入首轮实验结果；
4. 基于观测数据训练贝叶斯优化模型并推荐下一轮实验；
5. 查看全空间预测结果。

示例中的实验结果由一个虚拟响应函数生成，仅用于演示完整流程。真实实验时，把对应单元替换为实际测得的产率、选择性和电量/能耗即可。

In [3]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd

# 兼容从仓库根目录或 examples/electrochemistry 目录启动 Jupyter 的情况。
CWD = Path.cwd().resolve()
ROOT = next((path for path in [CWD, *CWD.parents] if (path / "edbo").exists()), CWD)
WORKDIR = ROOT / "examples" / "electrochemistry"
if not WORKDIR.exists():
    WORKDIR = CWD

sys.path.append(str(ROOT))

from edbo.plus.optimizer_botorch import EDBOplus

SCOPE_FILE = "electrochemistry_scope.csv"
ROUND0_FILE = "electrochemistry_round0.csv"
PRED_ROUND0_FILE = f"pred_{ROUND0_FILE}"

## 1. 定义电化学条件空间

`experiment_conditions.md` 给出的可优化变量包括：苯甲醇当量、催化剂种类与当量、电极种类、温度、电流和溶剂配比。

下面按该条件空间生成全组合候选集。若后续有真实候选清单，直接替换对应列表即可。

In [19]:
reaction_components = {
    "benzyl_alcohol_equiv": [1.5, 2.0, 2.5],
    "catalyst": ["TEMPO", "TEMPOL"],
    "catalyst_equiv": [0.15, 0.2, 0.25],
    "electrode": ["Al", "Zn", "Sn", "Ag"],
    "temperature_C": [25, 40, 50, 60],
    "current_mA": [25, 40, 55, 70, 85],
    "solvent_ratio": [0.2, 0.3, 0.4, 0.5],
}

In [20]:
scope = EDBOplus().generate_reaction_scope(
    components=reaction_components,
    directory=str(WORKDIR),
    filename=SCOPE_FILE,
    check_overwrite=False,
)

scope.head()

Generating a reaction scope...
The scope was generated and contains 5760 possible reactions!


,benzyl_alcohol_equiv,catalyst,catalyst_equiv,electrode,temperature_C,current_mA,solvent_ratio
0,1.5,TEMPO,0.15,Al,25,25,0.2
1,1.5,TEMPO,0.15,Al,25,25,0.3
2,1.5,TEMPO,0.15,Al,25,25,0.4
3,1.5,TEMPO,0.15,Al,25,25,0.5
4,1.5,TEMPO,0.15,Al,25,40,0.2


In [21]:
df_scope = pd.read_csv(WORKDIR / SCOPE_FILE)
print(f"条件空间共有 {len(df_scope)} 个候选实验。")
df_scope.sample(5, random_state=0)

条件空间共有 5760 个候选实验。


,benzyl_alcohol_equiv,catalyst,catalyst_equiv,electrode,temperature_C,current_mA,solvent_ratio
464,1.5,TEMPO,0.20,Zn,60,40,0.2
98,1.5,TEMPO,0.15,Zn,25,85,0.4
1155,1.5,TEMPOL,0.15,Sn,40,70,0.5
852,1.5,TEMPO,0.25,Sn,50,70,0.2
1995,2.0,TEMPO,0.15,Al,60,70,0.5


## 2. 没有观测数据时推荐初始实验

首次运行时，CSV 里还没有目标值。EDBO+ 会根据特征空间采样方法选择一批初始实验，并在 CSV 中加入目标列和 `priority` 列。

本示例优化 1 个目标：

- `yield_percent`：目标产物产率，越高越好；

- （后两个暂不纳入考虑）
- `selectivity_percent`：目标选择性，越高越好；
- `charge_F_per_mol`：单位底物通过电量，用作能耗/电化学成本代理，越低越好。

In [22]:
OBJECTIVES = ["yield_percent"]
OBJECTIVE_MODE = ["max"]
BATCH_SIZE = 6

initial_suggestions = EDBOplus().run(
    directory=str(WORKDIR),
    filename=SCOPE_FILE,
    objectives=OBJECTIVES,
    objective_mode=OBJECTIVE_MODE,
    batch=BATCH_SIZE,
    columns_features="all",
    init_sampling_method="cvt",
    seed=0,
)

initial_suggestions.query("priority == 1")

There are no experimental observations yet. Random samples will be drawn.
The following columns are categorical and will be encoded using One-Hot-Encoding: ['catalyst', 'electrode']
Generated 6 initial samples using cvt sampling (seed = 0). Run finished!


,benzyl_alcohol_equiv,catalyst,catalyst_equiv,electrode,temperature_C,current_mA,solvent_ratio,yield_percent,priority
2506,2.0,TEMPO,0.2,Ag,40,40,0.4,PENDING,1
2445,2.0,TEMPO,0.2,Sn,50,40,0.3,PENDING,1
2433,2.0,TEMPO,0.2,Sn,40,70,0.3,PENDING,1
3254,2.0,TEMPOL,0.2,Al,50,70,0.4,PENDING,1
5253,2.5,TEMPOL,0.2,Zn,50,70,0.3,PENDING,1
3230,2.0,TEMPOL,0.2,Al,40,55,0.4,PENDING,1


## 3. 录入首轮实验结果

真实实验中，完成 `priority == 1` 的实验后，把每个目标的 `PENDING` 改为实测值即可。

为了让教程可以从头跑通，下面用一个虚拟响应函数模拟首轮实验结果。它只代表“如何把结果写回 CSV”，不代表真实反应规律。

In [23]:
def virtual_electrochemistry_result(row, rng):
    """生成可复现的虚拟实验结果；真实项目中请替换为实测值。"""
    catalyst_bonus = {
        "TEMPO": 0,
        "TEMPOL": 6,
    }[row["catalyst"]]
    electrode_bonus = {
        "Al": 2,
        "Zn": 0,
        "Sn": 5,
        "Ag": 8,
    }[row["electrode"]]

    alcohol_penalty = -8 * abs(row["benzyl_alcohol_equiv"] - 2.0)
    catalyst_penalty = -40 * abs(row["catalyst_equiv"] - 0.2)
    temperature_penalty = -0.3 * abs(row["temperature_C"] - 50)
    current_penalty = -0.15 * abs(row["current_mA"] - 55)
    solvent_penalty = -30 * abs(row["solvent_ratio"] - 0.4)

    yield_percent = 52 + catalyst_bonus + electrode_bonus
    yield_percent += (
        alcohol_penalty
        + catalyst_penalty
        + temperature_penalty
        + current_penalty
        + solvent_penalty
    )
    yield_percent += rng.normal(0, 2.0)
    yield_percent = float(np.clip(yield_percent, 0, 100))

    selectivity_percent = 64 + 0.5 * catalyst_bonus + 0.4 * electrode_bonus
    selectivity_percent += -2.5 * max(row["benzyl_alcohol_equiv"] - 2.0, 0)
    selectivity_percent += -0.15 * abs(row["temperature_C"] - 50)
    selectivity_percent += -0.08 * abs(row["current_mA"] - 55)
    selectivity_percent += rng.normal(0, 1.5)
    selectivity_percent = float(np.clip(selectivity_percent, 0, 100))

    charge_F_per_mol = 2.5 - 0.015 * yield_percent
    charge_F_per_mol += 0.2 * (row["electrode"] in ["Al", "Zn"])
    charge_F_per_mol += 0.12 * (row["solvent_ratio"] <= 0.3)
    charge_F_per_mol += 0.004 * max(row["current_mA"] - 55, 0)
    charge_F_per_mol += rng.normal(0, 0.05)
    charge_F_per_mol = float(np.clip(charge_F_per_mol, 0.8, 3.5))

    return pd.Series(
        {
            "yield_percent": round(yield_percent, 1),
            "selectivity_percent": round(selectivity_percent, 1),
            "charge_F_per_mol": round(charge_F_per_mol, 2),
        }
    )

In [24]:
df_round0 = pd.read_csv(WORKDIR / SCOPE_FILE)
initial_mask = df_round0["priority"] == 1
rng = np.random.default_rng(42)

simulated_results = df_round0.loc[initial_mask].apply(
    lambda row: virtual_electrochemistry_result(row, rng), axis=1
)
df_round0.loc[initial_mask, OBJECTIVES] = simulated_results[OBJECTIVES]

df_round0.to_csv(WORKDIR / ROUND0_FILE, index=False)
df_round0.loc[
    initial_mask,
    [
        "benzyl_alcohol_equiv",
        "catalyst",
        "catalyst_equiv",
        "electrode",
        "temperature_C",
        "current_mA",
        "solvent_ratio",
        *OBJECTIVES,
    ],
]

,benzyl_alcohol_equiv,catalyst,catalyst_equiv,electrode,temperature_C,current_mA,solvent_ratio,yield_percent
0,2.0,TEMPO,0.2,Ag,40,40,0.4,55.4
1,2.0,TEMPO,0.2,Sn,50,40,0.3,53.6
2,2.0,TEMPO,0.2,Sn,40,70,0.3,49.0
3,2.0,TEMPOL,0.2,Al,50,70,0.4,56.0
4,2.5,TEMPOL,0.2,Zn,50,70,0.3,48.9
5,2.0,TEMPOL,0.2,Al,40,55,0.4,55.3


## 4. 用首轮观测数据推荐下一轮实验

现在 `electrochemistry_round0.csv` 中已经有一批非 `PENDING` 的观测值。再次运行 EDBO+ 时，它会训练代理模型，并为未测试条件分配新的优先级。

In [2]:
next_suggestions = EDBOplus().run(
    directory=str(WORKDIR),
    filename=ROUND0_FILE,
    objectives=OBJECTIVES,
    objective_mode=OBJECTIVE_MODE,
    batch=BATCH_SIZE,
    columns_features="all",
    init_sampling_method="cvt",
    seed=1,
)

next_suggestions.query("priority == 1")

NameError: name 'OBJECTIVES' is not defined

## 5. 查看全空间预测

当输入文件中包含观测值时，EDBO+ 会额外写出 `pred_<filename>`。该文件包含每个目标的预测均值、预测标准差和期望改进值，可用于理解模型为什么推荐某些实验。

In [26]:
df_predictions = pd.read_csv(WORKDIR / PRED_ROUND0_FILE)

prediction_columns = [
    "priority",
    "yield_percent_predicted_mean",
    "yield_percent_predicted_std_dev",
    "yield_percent_expected_improvement",
    "selectivity_percent_predicted_mean",
    "selectivity_percent_predicted_std_dev",
    "selectivity_percent_expected_improvement",
    "charge_F_per_mol_predicted_mean",
    "charge_F_per_mol_predicted_std_dev",
    "charge_F_per_mol_expected_improvement",
]

df_predictions[
    [
        "benzyl_alcohol_equiv",
        "catalyst",
        "catalyst_equiv",
        "electrode",
        "temperature_C",
        "current_mA",
        "solvent_ratio",
        *OBJECTIVES,
        *prediction_columns,
    ]
].head(12)

KeyError: "['selectivity_percent_predicted_mean', 'selectivity_percent_predicted_std_dev', 'selectivity_percent_expected_improvement', 'charge_F_per_mol_predicted_mean', 'charge_F_per_mol_predicted_std_dev', 'charge_F_per_mol_expected_improvement'] not in index"

## 6. 进入下一轮

实际优化时，重复以下循环即可：

1. 按 `priority == 1` 的条件做实验；
2. 将 `PENDING` 替换为真实实验结果；
3. 保存为新的 round CSV；
4. 再次运行 `EDBOplus().run(...)` 获取下一批推荐。

若目标只关注产率，可以把 `OBJECTIVES` 改为 `["yield_percent"]`、`OBJECTIVE_MODE` 改为 `["max"]`。若要加入成本、安全性、电压或反应时间，也可以增加目标列，并相应设置 `max` 或 `min`。